# Phase Transition Proof: Brittle Glass vs. Untethered Gel

Proves the two-regime hypothesis across **multiple architectures and modalities**:

- **Regime 1: "Brittle Glass"** -- manifold fractures internally, no rotation can fix it
- **Regime 2: "Untethered Gel"** -- manifold drifts globally but preserves internal structure

## Architectures Supported

| Architecture | Modality | Models / Scale | Perturbations |
|---|---|---|---|
| **ESM-2** | Protein | 8M - 15B params | aa_sub (1-10%), reverse |
| **Nucleotide Transformer** | DNA | 50M - 2.5B params | SNP (1-10%), reverse complement |
| **Caduceus** | DNA (SSM) | 0.5M - 7.7M params | SNP (1-10%), reverse complement |

## Two Tests

1. **Procrustes Ratio** -- Can a rigid rotation/translation align perturbed back to clean?
2. **Frozen Head** -- Does the MLM head still work on drifted embeddings?

## Prerequisites

Run after the relevant scaling/real-data experiments. Uses cached `.npy` embeddings.

---

In [ ]:
# Install & Imports
!pip install -q torch transformers matplotlib seaborn pandas scipy

import numpy as np
import os
import glob
from scipy.spatial import procrustes
from scipy.linalg import orthogonal_procrustes
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import torch
import transformers

np.random.seed(320)

# --- Patch transformers for Nucleotide Transformer compatibility ---
# NT v2 custom modeling code was written for transformers ~4.20-4.30.
# In transformers 5.x, several things were removed/moved.
# This monkey-patch restores everything so trust_remote_code works with NT models.
print("Applying transformers compatibility patches for NT...")

import torch.nn as nn
import transformers.pytorch_utils as _ptu
from transformers import PretrainedConfig

# Patch 1: Restore missing functions in pytorch_utils
if not hasattr(_ptu, 'find_pruneable_heads_and_indices'):
    def find_pruneable_heads_and_indices(heads, n_heads, head_size, already_pruned_heads):
        mask = torch.ones(n_heads, head_size)
        heads = set(heads) - already_pruned_heads
        for head in heads:
            head = head - sum(1 if h < head else 0 for h in already_pruned_heads)
            mask[head] = 0
        mask = mask.view(-1).contiguous().eq(1)
        index = torch.arange(len(mask))[mask].long()
        return heads, index
    _ptu.find_pruneable_heads_and_indices = find_pruneable_heads_and_indices

if not hasattr(_ptu, 'prune_linear_layer'):
    def prune_linear_layer(layer, index, dim=0):
        index = index.to(layer.weight.device)
        W = layer.weight.index_select(dim, index).clone().detach()
        if layer.bias is not None:
            b = layer.bias.clone().detach() if dim == 1 else layer.bias[index].clone().detach()
        new_size = list(layer.weight.size())
        new_size[dim] = len(index)
        new_layer = nn.Linear(new_size[1], new_size[0], bias=layer.bias is not None).to(layer.weight.device)
        new_layer.weight.requires_grad = False
        new_layer.weight.copy_(W.contiguous())
        new_layer.weight.requires_grad = True
        if layer.bias is not None:
            new_layer.bias.requires_grad = False
            new_layer.bias.copy_(b.contiguous())
            new_layer.bias.requires_grad = True
        return new_layer
    _ptu.prune_linear_layer = prune_linear_layer

# Patch 2: Restore missing PreTrainedModel methods
from transformers import PreTrainedModel as _PTM

if not hasattr(_PTM, 'get_head_mask'):
    def get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
        if head_mask is not None:
            if head_mask.dim() == 1:
                head_mask = head_mask.unsqueeze(0).unsqueeze(0).unsqueeze(-1).unsqueeze(-1)
                head_mask = head_mask.expand(num_hidden_layers, -1, -1, -1, -1)
            elif head_mask.dim() == 2:
                head_mask = head_mask.unsqueeze(1).unsqueeze(-1).unsqueeze(-1)
            head_mask = head_mask.to(dtype=next(self.parameters()).dtype)
            if is_attention_chunked:
                head_mask = head_mask.unsqueeze(-1)
        else:
            head_mask = [None] * num_hidden_layers
        return head_mask
    _PTM.get_head_mask = get_head_mask

if not hasattr(_PTM, 'get_extended_attention_mask'):
    def get_extended_attention_mask(self, attention_mask, input_shape, device=None, dtype=None):
        if attention_mask.dim() == 3:
            extended = attention_mask[:, None, :, :]
        elif attention_mask.dim() == 2:
            extended = attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape for attention_mask: {attention_mask.shape}")
        if dtype is None:
            dtype = next(self.parameters()).dtype
        extended = extended.to(dtype=dtype)
        extended = (1.0 - extended) * torch.finfo(dtype).min
        return extended
    _PTM.get_extended_attention_mask = get_extended_attention_mask

if not hasattr(_PTM, 'invert_attention_mask'):
    def invert_attention_mask(self, encoder_attention_mask):
        if encoder_attention_mask.dim() == 3:
            extended = encoder_attention_mask[:, None, :, :]
        elif encoder_attention_mask.dim() == 2:
            extended = encoder_attention_mask[:, None, None, :]
        else:
            raise ValueError(f"Wrong shape: {encoder_attention_mask.shape}")
        dtype = next(self.parameters()).dtype
        extended = extended.to(dtype=dtype)
        extended = (1.0 - extended) * torch.finfo(dtype).min
        return extended
    _PTM.invert_attention_mask = invert_attention_mask

# Patch 3: Fix all_tied_weights_keys / tie_weights compatibility
import transformers.modeling_utils as _mu
if not getattr(_mu, '_nt_patched', False):
    orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized
    def safe_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return orig_mark(self)
    _mu.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark

    orig_finalize = _mu.PreTrainedModel._finalize_load_state_dict
    @staticmethod
    def safe_finalize(model, load_config, load_info):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        orig_tie = model.tie_weights
        def _patched_tie(missing_keys=None, recompute_mapping=False, **kwargs):
            return orig_tie()
        model.tie_weights = _patched_tie
        return orig_finalize(model, load_config, load_info)
    _mu.PreTrainedModel._finalize_load_state_dict = safe_finalize

    _mu._nt_patched = True

# Patch 4: Restore missing default config attributes
_config_defaults = {
    'is_decoder': False,
    'add_cross_attention': False,
    'chunk_size_feed_forward': 0,
    'is_encoder_decoder': False,
    'tie_word_embeddings': True,
}
for attr, default in _config_defaults.items():
    if not hasattr(PretrainedConfig, attr):
        setattr(PretrainedConfig, attr, default)

print("✓ Transformers patched for NT compatibility")
print("Ready")


In [ ]:
# Configuration & Discovering Cached Embeddings
# Multi-architecture support: ESM-2, Nucleotide Transformer, Caduceus, etc.

import os

OUTPUT_BASE = './results/phase_transition_proof/'
RESULTS_DIR = OUTPUT_BASE + 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

CACHE_DIRS = {
    'DNABERT (Synthetic)': './results/dnabert_scaling_experiment/cache',
    'DNABERT (RealDNA)':   './results/dnabert_realdna_experiment/cache',
    'DNABERT-2 (Synthetic)':'./results/dnabert2_scaling_experiment/cache',
    'DNABERT-2 (RealDNA)': './results/dnabert2_realdna_experiment/cache',
    'HyenaDNA (Synthetic)':'./results/hyenadna_scaling_experiment/cache',
    'HyenaDNA (RealDNA)':  './results/hyenadna_realdna_experiment/cache',
    'GPN (Synthetic)':     './results/gpn_scaling_experiment/cache',
    'GPN (RealDNA)':       './results/gpn_realdna_experiment/cache',
    'ProtMamba (Synthetic)':'./results/protmamba_scaling_experiment_bins/cache',
    'ProtMamba (UniRef)':  './results/protmamba_uniref_experiment_bins/cache',
    'OpenFold (Synthetic)': './results/openfold_scaling_experiment/cache',
    'OpenFold (UniRef)':    './results/openfold_uniref_experiment/cache',
    'ESM-2 (UniRef)':   './results/esm2_uniref/cache',
    'ESM-2 (Synthetic)':'./results/esm2_scaling_experiment/cache',
    'NT (RealDNA)':     './results/nt_realdna_experiment/cache',
    'NT (Synthetic)':   './results/nt_scaling_experiment/cache',
    'Caduceus (Synthetic)':'./results/caduceus_scaling_experiment/cache',
    'Caduceus (RealDNA)': './results/caduceus_realdna_experiment/cache',
}

MODEL_DEFS = {
    # --- DNABERT ---
    'DNABERT (Synthetic)': {
        'models': [
            ('zhihan1996/DNA_bert_3', 'DNABERT-3mer', 86),
            ('zhihan1996/DNA_bert_6', 'DNABERT-6mer', 90),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'DNABERT',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'DNABERT (RealDNA)': {
        'models': [
            ('zhihan1996/DNA_bert_3', 'DNABERT-3mer', 86),
            ('zhihan1996/DNA_bert_6', 'DNABERT-6mer', 90),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'DNABERT',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },

    # --- DNABERT-2 ---
    'DNABERT-2 (Synthetic)': {
        'models': [('zhihan1996/DNABERT-2-117M', 'DNABERT-2', 117)],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'DNABERT-2',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'DNABERT-2 (RealDNA)': {
        'models': [('zhihan1996/DNABERT-2-117M', 'DNABERT-2', 117)],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'DNABERT-2',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },

    # --- HyenaDNA ---
    'HyenaDNA (Synthetic)': {
        'models': [
            ('LongSafari/hyenadna-tiny-1k-seqlen-hf', 'Hyena-tiny', 0.5),
            ('LongSafari/hyenadna-small-32k-seqlen-hf', 'Hyena-small', 3.0),
            ('LongSafari/hyenadna-medium-160k-seqlen-hf', 'Hyena-medium', 14.0),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'HyenaDNA',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'HyenaDNA (RealDNA)': {
        'models': [
            ('LongSafari/hyenadna-tiny-1k-seqlen-hf', 'Hyena-tiny', 0.5),
            ('LongSafari/hyenadna-small-32k-seqlen-hf', 'Hyena-small', 3.0),
            ('LongSafari/hyenadna-medium-160k-seqlen-hf', 'Hyena-medium', 14.0),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'HyenaDNA',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },

    # --- GPN ---
    'GPN (Synthetic)': {
        'models': [('songlab/gpn-brassicales', 'GPN-Brassicales', 66)],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'GPN',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'GPN (RealDNA)': {
        'models': [
            ('songlab/PhyloGPN', 'PhyloGPN', 83),
            ('songlab/gpn-brassicales', 'GPN-Brassicales', 66),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'GPN',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },

    # --- ProtMamba ---
    'ProtMamba (Synthetic)': {
        'models': [('protmamba_L400', 'PM-400', 400), ('protmamba_L800', 'PM-800', 800)],
        'perturbations': ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse'],
        'arch': 'ProtMamba',
        'name_fn': lambda m: m.lower(),
        'suffixes': ['_'],
    },
    'ProtMamba (UniRef)': {
        'models': [('ProtMamba', 'ProtMamba', 484)],
        'perturbations': ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse'],
        'arch': 'ProtMamba',
        'name_fn': lambda m: 'protmamba_uniref_bin',
        'suffixes': ['100_400_', '_'],
    },

    # --- OpenFold ---
    'OpenFold (Synthetic)': {
        'models': [('openfold_L100', 'OF-100', 100), ('openfold_L500', 'OF-500', 500)],
        'perturbations': ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse'],
        'arch': 'OpenFold',
        'name_fn': lambda m: m.lower(),
        'suffixes': ['_'],
    },
    'OpenFold (UniRef)': {
        'models': [('openfold_uniref', 'OF-UniRef', 93)],
        'perturbations': ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse'],
        'arch': 'OpenFold',
        'name_fn': lambda m: 'openfold_uniref',
        'suffixes': ['_'],
    },

    # --- ESM-2 ---
    'ESM-2 (UniRef)': {
        'models': [
            ('facebook/esm2_t6_8M_UR50D',    'ESM2-8M',    8),
            ('facebook/esm2_t12_35M_UR50D',   'ESM2-35M',   35),
            ('facebook/esm2_t30_150M_UR50D',  'ESM2-150M',  150),
            ('facebook/esm2_t33_650M_UR50D',  'ESM2-650M',  650),
            ('facebook/esm2_t36_3B_UR50D',    'ESM2-3B',    3000),
            ('facebook/esm2_t48_15B_UR50D',   'ESM2-15B',   15000),
        ],
        'perturbations': ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse'],
        'arch': 'ESM-2',
        'name_fn': lambda m: m.replace('/', '_'),
        'suffixes': ['_uniref_', '_'],
    },
    'ESM-2 (Synthetic)': {
        'models': [
            ('facebook/esm2_t6_8M_UR50D',    'ESM2-8M',    8),
            ('facebook/esm2_t12_35M_UR50D',   'ESM2-35M',   35),
            ('facebook/esm2_t30_150M_UR50D',  'ESM2-150M',  150),
            ('facebook/esm2_t33_650M_UR50D',  'ESM2-650M',  650),
            ('facebook/esm2_t36_3B_UR50D',    'ESM2-3B',    3000),
            ('facebook/esm2_t48_15B_UR50D',   'ESM2-15B',   15000),
        ],
        'perturbations': ['aa_sub_1pct', 'aa_sub_2pct', 'aa_sub_5pct', 'aa_sub_10pct', 'reverse'],
        'arch': 'ESM-2',
        'name_fn': lambda m: m.replace('/', '_'),
        'suffixes': ['_'],
    },

    # --- NT ---
    'NT (RealDNA)': {
        'models': [
            ('InstaDeepAI/nucleotide-transformer-v2-50m-multi-species',  'NT-50M',   55.9),
            ('InstaDeepAI/nucleotide-transformer-v2-100m-multi-species', 'NT-100M',  113.2),
            ('InstaDeepAI/nucleotide-transformer-v2-250m-multi-species', 'NT-250M',  252.5),
            ('InstaDeepAI/nucleotide-transformer-v2-500m-multi-species', 'NT-500M',  498.5),
            ('InstaDeepAI/nucleotide-transformer-2.5b-multi-species',    'NT-2.5B',  2532.1),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'NT',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },
    'NT (Synthetic)': {
        'models': [
            ('InstaDeepAI/nucleotide-transformer-v2-50m-multi-species',  'NT-50M',   55.9),
            ('InstaDeepAI/nucleotide-transformer-v2-100m-multi-species', 'NT-100M',  113.2),
            ('InstaDeepAI/nucleotide-transformer-v2-250m-multi-species', 'NT-250M',  252.5),
            ('InstaDeepAI/nucleotide-transformer-v2-500m-multi-species', 'NT-500M',  498.5),
            ('InstaDeepAI/nucleotide-transformer-2.5b-multi-species',    'NT-2.5B',  2532.1),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'NT',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },

    # --- Caduceus ---
    'Caduceus (Synthetic)': {
        'models': [
            ('kuleshov-group/caduceus-ps_seqlen-1k_d_model-118_n_layer-4_lr-8e-3', 'Cad-0.5M',  0.5),
            ('kuleshov-group/caduceus-ps_seqlen-1k_d_model-256_n_layer-4_lr-8e-3', 'Cad-1.9M',  1.93),
            ('kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16',      'Cad-7.7M',  7.73),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'Caduceus',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_'),
        'suffixes': ['_'],
    },
    'Caduceus (RealDNA)': {
        'models': [
            ('kuleshov-group/caduceus-ps_seqlen-1k_d_model-118_n_layer-4_lr-8e-3', 'Cad-0.5M',  0.5),
            ('kuleshov-group/caduceus-ps_seqlen-1k_d_model-256_n_layer-4_lr-8e-3', 'Cad-1.9M',  1.93),
            ('kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16',      'Cad-7.7M',  7.73),
        ],
        'perturbations': ['snp_1pct', 'snp_2pct', 'snp_5pct', 'snp_10pct', 'reverse_complement'],
        'arch': 'Caduceus',
        'name_fn': lambda m: m.replace('/', '_').replace('-', '_') + '_realdna',
        'suffixes': ['_'],
    },

}

def find_cache_path(base_dir, model_safe, suffix_list, pert):
    for suffix in suffix_list:
        path = os.path.join(base_dir, f"{model_safe}{suffix}{pert}.npy")
        if os.path.exists(path):
            return path
    return None

available_models = []
for exp_name, info in MODEL_DEFS.items():
    arch = info['arch']
    cdir = CACHE_DIRS.get(exp_name)
    if not cdir or not os.path.exists(cdir):
        continue
    
    for model_id, label, size in info['models']:
        safe_name = info['name_fn'](model_id)
        available_models.append({
            'experiment': exp_name,
            'arch': arch,
            'model_id': model_id,
            'label': label,
            'size_M': size,
            'cache_dir': cdir,
            'safe_name': safe_name,
            'suffixes': info['suffixes'],
            'perturbations': info['perturbations'],
        })

print(f"Scanned {len(available_models)} available models across {len(MODEL_DEFS)} experiments.")


In [ ]:
# TEST 1 -- The Procrustes Ratio (Multi-Architecture)
#
# For each model+perturbation:
#   Raw Error = ||X - X'||_F  (Frobenius norm of difference)
#   Aligned Error = Procrustes distance (best rotation+translation+scaling)
#   Ratio = Aligned Error / Raw Error
#
# Prediction:
#   Small models (stable): Both low, ratio ~1 (little error to fix)
#   Mid-range (brittle): Ratio ~0.8-1.0 (can't fix fractures with rotation)
#   Largest (gel): Ratio ~0.1-0.3 (rotation fixes the global drift)

def compute_procrustes_ratio(X_clean, X_pert, n_samples=10000):
    """
    Compute the Procrustes ratio: how much error is "alignable".
    """
    if X_clean.shape[0] > n_samples:
        idx = np.random.choice(X_clean.shape[0], n_samples, replace=False)
        X_clean = X_clean[idx]
        X_pert = X_pert[idx]

    X_c = X_clean - X_clean.mean(axis=0)
    X_p = X_pert - X_pert.mean(axis=0)

    raw_error = np.linalg.norm(X_c - X_p, 'fro') / np.sqrt(X_c.shape[0])

    scale_c = np.linalg.norm(X_c, 'fro')
    scale_p = np.linalg.norm(X_p, 'fro')
    X_cn = X_c / scale_c
    X_pn = X_p / scale_p

    R, _ = orthogonal_procrustes(X_pn, X_cn)
    X_aligned = X_p @ R

    s = np.trace(X_c.T @ X_aligned) / np.trace(X_aligned.T @ X_aligned)
    X_aligned_scaled = X_aligned * s

    aligned_error = np.linalg.norm(X_c - X_aligned_scaled, 'fro') / np.sqrt(X_c.shape[0])

    ratio = aligned_error / raw_error if raw_error > 1e-10 else 1.0
    reduction = 1.0 - ratio

    return {
        'raw_error': raw_error,
        'aligned_error': aligned_error,
        'ratio': ratio,
        'reduction_pct': reduction * 100,
        'optimal_scale': s,
    }


# Run Procrustes on ALL available models x perturbations
print('=' * 70)
print('TEST 1: PROCRUSTES RATIO (Brittle vs. Gel)  --  All Architectures')
print('=' * 70)

procrustes_rows = []

for minfo in available_models:
    label = minfo['label']
    arch  = minfo['arch']
    size  = minfo['size_M']
    cdir  = minfo['cache_dir']
    sname = minfo['safe_name']
    sfxs  = minfo['suffixes']
    perts = minfo['perturbations']

    clean_path = find_cache_path(cdir, sname, sfxs, 'clean')
    if clean_path is None:
        continue

    print(f'\n--- [{arch}] {label} ({size}M params)  [{minfo["experiment"]}] ---')
    X_clean = np.load(clean_path)

    for pert in perts:
        pert_path = find_cache_path(cdir, sname, sfxs, pert)
        if pert_path is None:
            continue

        X_pert = np.load(pert_path)
        result = compute_procrustes_ratio(X_clean, X_pert)

        procrustes_rows.append({
            'arch': arch,
            'experiment': minfo['experiment'],
            'model': label,
            'size_M': size,
            'perturbation': pert,
            'raw_error': result['raw_error'],
            'aligned_error': result['aligned_error'],
            'ratio': result['ratio'],
            'reduction_pct': result['reduction_pct'],
            'optimal_scale': result['optimal_scale'],
        })

        print(f'  {pert:>20}: raw={result["raw_error"]:.4f}  aligned={result["aligned_error"]:.4f}  '
              f'ratio={result["ratio"]:.3f}  reduction={result["reduction_pct"]:.1f}%  '
              f'scale={result["optimal_scale"]:.4f}')

df_proc = pd.DataFrame(procrustes_rows)
df_proc.to_csv(f'{RESULTS_DIR}/procrustes_ratio.csv', index=False)
for a in df_proc['arch'].unique():
    df_a = df_proc[df_proc['arch'] == a]
    s_arch = str(a).replace(' ', '_').replace('-', '_').lower()
    df_a.to_csv(f'{RESULTS_DIR}/procrustes_ratio_{s_arch}.csv', index=False)
    print(f'Saved architecture split: {RESULTS_DIR}/procrustes_ratio_{s_arch}.csv')
print(f'\nSaved {len(df_proc)} rows to {RESULTS_DIR}/procrustes_ratio.csv')
print(f'Architectures: {sorted(df_proc["arch"].unique().tolist())}')

In [ ]:
# Procrustes Visualization (Multi-Architecture)

plt.rcParams.update({'font.size': 11})

# Colour palette: one colour per architecture
ARCH_COLORS = {
    'DNABERT':   '#EAB308',
    'DNABERT-2': '#F59E0B',
    'HyenaDNA':  '#059669',
    'GPN':       '#0891B2',
    'ProtMamba': '#DB2777',
    'OpenFold':  '#4F46E5',
    'ESM-2':     '#2563EB',
    'NT':        '#DC2626',
    'Caduceus':  '#16A34A',
}
ARCH_MARKERS = {
    'DNABERT':   '^',
    'DNABERT-2': 'v',
    'HyenaDNA':  '<',
    'GPN':       '>',
    'ProtMamba': 'H',
    'OpenFold':  '*',
    'ESM-2':     'o',
    'NT':        'D',
    'Caduceus':  's',
}

# Aggregate by (arch, model, size_M)
proc_agg = df_proc.groupby(['arch', 'model', 'size_M']).agg(
    ratio_mean=('ratio', 'mean'),
    ratio_std=('ratio', 'std'),
    raw_mean=('raw_error', 'mean'),
    aligned_mean=('aligned_error', 'mean'),
    reduction_mean=('reduction_pct', 'mean'),
).reset_index().sort_values(['arch', 'size_M'])

archs_present = proc_agg['arch'].unique()

fig, axes = plt.subplots(1, 3, figsize=(19, 6))

# Panel A: Raw vs Aligned Error by architecture
ax = axes[0]
for arch in archs_present:
    sub = proc_agg[proc_agg['arch'] == arch]
    c = ARCH_COLORS.get(arch, '#6B7280')
    m = ARCH_MARKERS.get(arch, 'o')
    ax.plot(sub['size_M'], sub['raw_mean'], marker=m, linestyle='--',
            color=c, linewidth=1.5, markersize=7, alpha=0.5, label=f'{arch} raw')
    ax.plot(sub['size_M'], sub['aligned_mean'], marker=m, linestyle='-',
            color=c, linewidth=2.5, markersize=9, label=f'{arch} aligned')
ax.set_xscale('log')
ax.set_xlabel('Parameters (M)')
ax.set_ylabel('Frobenius Error (per sample)')
ax.set_title('A. Raw vs. Aligned Error', fontweight='bold')
ax.legend(fontsize=8, ncol=2)
ax.grid(True, alpha=0.15)

# Panel B: Procrustes Ratio bars grouped by architecture
ax = axes[1]
x_pos = 0
tick_positions, tick_labels = [], []
for arch in archs_present:
    sub = proc_agg[proc_agg['arch'] == arch].sort_values('size_M')
    c = ARCH_COLORS.get(arch, '#6B7280')
    for _, row in sub.iterrows():
        ax.bar(x_pos, row['ratio_mean'], yerr=row['ratio_std'],
               color=c, edgecolor='white', linewidth=1.2, capsize=3, width=0.7, zorder=3)
        ax.text(x_pos, row['ratio_mean'] + row['ratio_std'] + 0.03, f'{row["ratio_mean"]:.2f}',
                ha='center', fontsize=7.5, fontweight='bold')
        tick_positions.append(x_pos)
        tick_labels.append(row['model'])
        x_pos += 1
    x_pos += 0.5  # gap between architectures
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels, fontsize=8, rotation=45, ha='right')
ax.set_ylabel('Procrustes Ratio (aligned / raw)')
ax.set_title('B. Procrustes Ratio (all models)', fontweight='bold')
ax.set_ylim(0, 1.15)
ax.axhline(y=1.0, color='#94A3B8', linestyle=':', linewidth=1, alpha=0.5)
ax.grid(True, alpha=0.15, axis='y')
# Legend patches for architecture colour
from matplotlib.patches import Patch
legend_patches = [Patch(facecolor=ARCH_COLORS.get(a, '#6B7280'), label=a) for a in archs_present]
ax.legend(handles=legend_patches, fontsize=9)

# Panel C: Per-perturbation Procrustes ratio (pick smallest + largest per arch)
ax = axes[2]
key_models = []
for arch in archs_present:
    sub = proc_agg[proc_agg['arch'] == arch].sort_values('size_M')
    if len(sub) >= 2:
        key_models.append(sub.iloc[0])   # smallest
        key_models.append(sub.iloc[-1])  # largest
    elif len(sub) == 1:
        key_models.append(sub.iloc[0])

if key_models:
    key_labels = [r['model'] for r in key_models]
    key_df = df_proc[df_proc['model'].isin(key_labels)]
    pivot = key_df.pivot_table(values='ratio', index='perturbation', columns='model')
    # Reorder columns by (arch, size)
    col_order = [r['model'] for r in key_models if r['model'] in pivot.columns]
    pivot = pivot[[c for c in col_order if c in pivot.columns]]
    pivot.plot(kind='bar', ax=ax, width=0.75, edgecolor='white', linewidth=1)
    ax.set_ylabel('Procrustes Ratio')
    ax.set_title('C. Ratio by Perturbation (Key Models)', fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.15, axis='y')
    ax.set_ylim(0, 1.15)

fig.suptitle('Procrustes Analysis Across Architectures: ESM-2, NT, Caduceus',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/procrustes_analysis.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved {RESULTS_DIR}/procrustes_analysis.png')

In [ ]:
# TEST 2 -- The Frozen Head Test (Multi-Architecture)
#
# Architecture-aware loading:
#   - ESM-2, NT: AutoModelForMaskedLM (native MLM heads)
#   - DNABERT (v1 3mer/6mer): AutoModelForMaskedLM (BERT-based, native MLM)
#   - DNABERT-2: Custom BertForMaskedLM with trust_remote_code + config patch
#   - HyenaDNA: AutoModelForCausalLM (causal LM, not MLM -- test next-token stability)
#   - GPN: Custom ConvNet -- requires `gpn` package or manual model loading
#   - Caduceus: Custom Mamba-based -- requires mamba_ssm or trust_remote_code
#
# ProtMamba and OpenFold are tested in their own Ghost Detection notebooks.

import torch
import gc
import sys
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoModelForCausalLM, AutoModel
import numpy as np

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

# ---- Inline perturbation helpers ----
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')
DNA_BASES = list('ACGT')
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

_rng_frozen = np.random.default_rng(320)

def _mutate_protein(seq, rate):
    s = list(seq)
    n = max(1, int(len(s) * rate))
    pos = _rng_frozen.choice(len(s), size=n, replace=False)
    for p in pos:
        alts = [a for a in AMINO_ACIDS if a != s[p]]
        s[p] = _rng_frozen.choice(alts)
    return ''.join(s)

def _mutate_dna(seq, rate):
    s = list(seq)
    n = max(1, int(len(s) * rate))
    pos = _rng_frozen.choice(len(s), size=n, replace=False)
    for p in pos:
        alts = [b for b in DNA_BASES if b != s[p]]
        s[p] = _rng_frozen.choice(alts)
    return ''.join(s)

def _reverse_complement(seq):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(seq))

def string_kmerize(seq, k=6):
    return " ".join([seq[i:i+k] for i in range(len(seq) - k + 1)])

# ---- Architecture-aware perturbation sets ----
FROZEN_PERTS = {
    'ESM-2': [
        ('aa_sub_1pct',  lambda s: _mutate_protein(s, 0.01)),
        ('aa_sub_5pct',  lambda s: _mutate_protein(s, 0.05)),
        ('aa_sub_10pct', lambda s: _mutate_protein(s, 0.10)),
        ('reverse',      lambda s: s[::-1]),
    ],
}
for dna_arch in ['NT', 'DNABERT', 'DNABERT-2', 'HyenaDNA', 'GPN', 'Caduceus']:
    FROZEN_PERTS[dna_arch] = [
        ('snp_1pct',            lambda s: _mutate_dna(s, 0.01)),
        ('snp_5pct',            lambda s: _mutate_dna(s, 0.05)),
        ('snp_10pct',           lambda s: _mutate_dna(s, 0.10)),
        ('reverse_complement',  lambda s: _reverse_complement(s)),
    ]


# ---- Architecture-specific model loaders ----

def _load_dnabert2(model_name, device, trust_remote):
    """
    DNABERT-2 uses custom code that expects config.pad_token_id.
    Patch the config before model init.
    """
    from transformers import AutoConfig
    config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    # Patch missing pad_token_id
    if not hasattr(config, 'pad_token_id') or config.pad_token_id is None:
        config.pad_token_id = 0
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or '[PAD]'
    model = AutoModelForMaskedLM.from_pretrained(
        model_name, config=config, trust_remote_code=True
    ).to(device).eval()
    return model, tokenizer, 'mlm'


def _load_hyenadna(model_name, device, trust_remote):
    """
    HyenaDNA is a causal (autoregressive) model, NOT an MLM.
    AutoModelForMaskedLM will fail because HyenaConfig is not in the MLM registry.
    Load as CausalLM instead and test next-token prediction stability.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token or '[PAD]'
    model = AutoModelForCausalLM.from_pretrained(
        model_name, trust_remote_code=True
    ).to(device).eval()
    return model, tokenizer, 'causal'


def _load_gpn(model_name, device, trust_remote):
    """
    GPN uses a custom ConvNet architecture not in the AutoModel registry.
    Requires the `gpn` package from songlab.
    Falls back to embedding-only test if gpn not installed.
    """
    try:
        import gpn.model
        from transformers import AutoConfig
        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        model = gpn.model.ConvNetForMaskedLM.from_pretrained(
            model_name, config=config
        ).to(device).eval()
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = '[PAD]'
        return model, tokenizer, 'mlm'
    except ImportError:
        raise RuntimeError(
            "GPN requires `pip install git+https://github.com/songlab-cal/gpn.git`. "
            "Install it and re-run, or skip GPN in the frozen head test."
        )


def _load_caduceus(model_name, device, trust_remote):
    """
    Caduceus is Mamba-based. It exposes an MLM head via trust_remote_code.
    Requires mamba_ssm and causal-conv1d packages.
    """
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token or '[PAD]'
        model = AutoModelForMaskedLM.from_pretrained(
            model_name, trust_remote_code=True
        ).to(device).eval()
        return model, tokenizer, 'mlm'
    except Exception as e:
        if 'mamba_ssm' in str(e).lower() or 'causal_conv1d' in str(e).lower():
            raise RuntimeError(
                f"Caduceus requires mamba_ssm and causal-conv1d: "
                f"pip install mamba_ssm causal-conv1d. Original error: {e}"
            )
        raise


def _load_default_mlm(model_name, device, trust_remote):
    """Default: ESM-2, NT, DNABERT-v1 -- standard AutoModelForMaskedLM."""
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=trust_remote)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token or '[PAD]'
        if getattr(tokenizer, 'pad_token', None) is None:
            tokenizer.add_special_tokens({'pad_token': '[PAD]'})
    model = AutoModelForMaskedLM.from_pretrained(
        model_name, trust_remote_code=trust_remote
    ).to(device).eval()
    return model, tokenizer, 'mlm'


# Dispatch table
MODEL_LOADERS = {
    'DNABERT-2': _load_dnabert2,
    'HyenaDNA':  _load_hyenadna,
    'GPN':       _load_gpn,
    'Caduceus':  _load_caduceus,
}


def frozen_head_test(model_name, label, sequences, arch='ESM-2',
                     n_test=500, batch_size=8, trust_remote=False):
    """
    Test: does the LM head still produce correct predictions
    when fed perturbed sequences?

    Uses architecture-specific loaders to handle non-standard models.
    For causal models (HyenaDNA), tests next-token prediction stability.
    """
    print(f'\nLoading {model_name} (with LM head)...')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    # Architecture-aware loading
    loader = MODEL_LOADERS.get(arch, _load_default_mlm)
    model, tokenizer, lm_type = loader(model_name, device, trust_remote)

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f'  Params: {n_params:.1f}M | Device: {device} | LM type: {lm_type}')

    test_seqs = sequences[:n_test]

    @torch.no_grad()
    def get_logits(seqs):
        all_logits, all_tokens = [], []
        for i in range(0, len(seqs), batch_size):
            batch = seqs[i:i + batch_size]

            # Format inputs based on architecture
            if arch == 'DNABERT':
                k = 6 if '6' in label else 3
                batch = [string_kmerize(s, k=k) for s in batch]

            tokens = tokenizer(batch, return_tensors='pt', padding=True,
                             truncation=True, max_length=512)
            tokens = {k: v.to(device) for k, v in tokens.items()}

            outputs = model(**tokens)
            all_logits.append(outputs.logits.cpu())
            all_tokens.append(tokens['input_ids'].cpu())

        return torch.cat(all_logits, dim=0), torch.cat(all_tokens, dim=0)

    print('  Computing clean logits...')
    try:
        logits_clean, tokens_clean = get_logits(test_seqs)
    except Exception as e:
        print(f"  FAILED during clean logits: {e}")
        del model, tokenizer
        cleanup_gpu()
        raise e

    results = {}
    perturbations = FROZEN_PERTS.get(arch, FROZEN_PERTS['ESM-2'])

    for pert_name, pert_fn in perturbations:
        perturbed_seqs = [pert_fn(s) for s in test_seqs]
        print(f'  Computing {pert_name} logits...')
        logits_pert, tokens_pert = get_logits(perturbed_seqs)

        pred_clean = logits_clean.argmax(dim=-1)
        pred_pert = logits_pert.argmax(dim=-1)

        # For causal LMs, tokens may differ in length after perturbation.
        # Align on the shorter length.
        min_len = min(pred_clean.shape[1], pred_pert.shape[1])
        pred_clean_t = pred_clean[:, :min_len]
        pred_pert_t = pred_pert[:, :min_len]
        tokens_clean_t = tokens_clean[:, :min_len]
        logits_clean_t = logits_clean[:, :min_len, :]
        logits_pert_t = logits_pert[:, :min_len, :]

        mask = (tokens_clean_t != tokenizer.pad_token_id)

        if mask.sum() > 0:
            agreement = (pred_clean_t[mask] == pred_pert_t[mask]).float().mean().item()

            probs_clean = torch.softmax(logits_clean_t, dim=-1)
            probs_pert = torch.softmax(logits_pert_t, dim=-1)

            # Clamp to avoid log(0)
            kl = torch.nn.functional.kl_div(
                probs_pert[mask].clamp(min=1e-10).log(),
                probs_clean[mask],
                reduction='batchmean'
            ).item()

            conf_clean = probs_clean.max(dim=-1).values[mask].mean().item()
            conf_pert = probs_pert.max(dim=-1).values[mask].mean().item()
        else:
            agreement, kl, conf_clean, conf_pert = 0, 0, 0, 0

        results[pert_name] = {
            'agreement': agreement,
            'kl_divergence': kl,
            'conf_clean': conf_clean,
            'conf_pert': conf_pert,
            'conf_drop': conf_clean - conf_pert,
        }

        print(f'    agreement={agreement:.4f}  KL={kl:.4f}  '
              f'conf_clean={conf_clean:.4f}  conf_pert={conf_pert:.4f}')

    del model, tokenizer
    cleanup_gpu()
    return results

print('Frozen Head test function ready (architecture-specific loaders)')


In [ ]:
# Run Frozen Head Test (Multi-Architecture)
import pandas as pd

rng_seq = np.random.default_rng(320)
protein_sequences = [''.join(rng_seq.choice(list('ACDEFGHIKLMNPQRSTVWY'), size=250)) for _ in range(500)]
dna_sequences = [''.join(rng_seq.choice(list('ACGT'), size=500)) for _ in range(500)]
print(f'Generated {len(protein_sequences)} protein + {len(dna_sequences)} DNA test sequences')

# ---- Check optional dependencies ----
def _check_deps():
    """Report which optional packages are available."""
    deps = {}
    for pkg in ['mamba_ssm', 'causal_conv1d', 'gpn']:
        try:
            __import__(pkg)
            deps[pkg] = True
        except ImportError:
            deps[pkg] = False
    return deps

opt_deps = _check_deps()
print(f'\nOptional dependencies:')
for pkg, avail in opt_deps.items():
    status = 'installed' if avail else 'NOT FOUND (models needing this will be skipped)'
    print(f'  {pkg}: {status}')

# Models that need special deps
NEEDS_DEPS = {
    'Caduceus': ['mamba_ssm', 'causal_conv1d'],
    'GPN': ['gpn'],
}

# ---- Compile models to test ----
KEY_FROZEN_MODELS = []
hf_architectures = ['DNABERT', 'DNABERT-2', 'HyenaDNA', 'GPN', 'ESM-2', 'NT', 'Caduceus']

for arch in hf_architectures:
    # Check if required deps are available
    required = NEEDS_DEPS.get(arch, [])
    missing = [d for d in required if not opt_deps.get(d, False)]
    if missing:
        print(f'\n  Skipping {arch}: missing {", ".join(missing)}')
        continue

    group_key = f'{arch} (Synthetic)'
    if group_key not in MODEL_DEFS:
        group_key = f'{arch}'
    if group_key not in MODEL_DEFS:
        continue

    for model_name, label, size in MODEL_DEFS[group_key]['models']:
        trust = arch in ['NT', 'HyenaDNA', 'GPN', 'DNABERT-2', 'Caduceus']
        bs = 8
        if size > 500: bs = 4
        if size > 2000: bs = 2
        if size > 10000: bs = 1
        KEY_FROZEN_MODELS.append((model_name, label, size, arch, trust, bs))

print(f'\nWill test {len(KEY_FROZEN_MODELS)} models')
print('=' * 70)
print('TEST 2: FROZEN HEAD (MLM Prediction Stability) -- All Architectures')
print('=' * 70)

frozen_results = {}
frozen_rows = []

for model_name, label, size, arch, trust_remote, bs in KEY_FROZEN_MODELS:
    print(f'\n{"="*70}')
    print(f'[{arch}] {label} ({size}M params)')
    print('=' * 70)

    seqs = dna_sequences if arch not in ['ESM-2', 'ProtMamba', 'OpenFold'] else protein_sequences

    try:
        results = frozen_head_test(
            model_name, label, seqs,
            arch=arch, n_test=500, batch_size=bs, trust_remote=trust_remote
        )
        frozen_results[label] = results

        for pert_name, metrics in results.items():
            frozen_rows.append({
                'arch': arch,
                'model': label,
                'size_M': size,
                'perturbation': pert_name,
                **metrics,
            })
    except Exception as e:
        print(f'  ERROR RUNNING {label}: {e}')
        print(f'  Skipping.')
        cleanup_gpu()

# Custom notice
print('\n' + '='*70)
print('NOTICE: ProtMamba and OpenFold use custom loading routines.')
print('Run their Frozen Head / Ghost Detection tests in their respective notebooks.')
print('='*70 + '\n')

df_frozen = pd.DataFrame(frozen_rows)
if len(df_frozen) > 0:
    df_frozen.to_csv(f'{RESULTS_DIR}/frozen_head_test.csv', index=False)
    print(f'\nSaved to {RESULTS_DIR}/frozen_head_test.csv')
    print(f'Architectures tested: {sorted(df_frozen["arch"].unique().tolist())}')
    print('\nFull results:')
    print(df_frozen.to_string(index=False))
else:
    print('\nNo models completed successfully. Check errors above.')


In [ ]:
# Frozen Head Visualization (Multi-Architecture)

if len(df_frozen) > 0:
    archs_frozen = sorted(df_frozen['arch'].unique())
    n_archs = len(archs_frozen)

    fig, axes = plt.subplots(n_archs, 3, figsize=(17, 5.5 * n_archs), squeeze=False)

    # Perturbation display names
    def _pert_label(p):
        return (p.replace('aa_sub_', '').replace('snp_', 'SNP ')
                 .replace('pct', '%').replace('reverse_complement', 'RC')
                 .replace('reverse', 'Reverse'))

    PERT_COLORS_ESM = {'aa_sub_1pct': '#60A5FA', 'aa_sub_5pct': '#34D399',
                       'aa_sub_10pct': '#FBBF24', 'reverse': '#F87171'}
    PERT_COLORS_NT = {'snp_1pct': '#60A5FA', 'snp_5pct': '#34D399',
                      'snp_10pct': '#FBBF24', 'reverse_complement': '#F87171'}

    for row_idx, arch in enumerate(archs_frozen):
        df_arch = df_frozen[df_frozen['arch'] == arch].copy()
        perts = sorted(df_arch['perturbation'].unique())
        pcols = PERT_COLORS_NT if arch == 'NT' else PERT_COLORS_ESM

        # Panel A: Agreement rate
        ax = axes[row_idx, 0]
        for pert in perts:
            d = df_arch[df_arch['perturbation'] == pert].sort_values('size_M')
            c = pcols.get(pert, '#6B7280')
            ax.plot(d['size_M'], d['agreement'], 'o-', color=c,
                    linewidth=2, markersize=8, label=_pert_label(pert))
        ax.set_xscale('log')
        ax.set_xlabel('Parameters (M)')
        ax.set_ylabel('Top-1 Agreement')
        ax.set_title(f'{arch}: MLM Agreement', fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.15)

        # Panel B: KL Divergence
        ax = axes[row_idx, 1]
        for pert in perts:
            d = df_arch[df_arch['perturbation'] == pert].sort_values('size_M')
            c = pcols.get(pert, '#6B7280')
            ax.plot(d['size_M'], d['kl_divergence'], 'o-', color=c,
                    linewidth=2, markersize=8, label=_pert_label(pert))
        ax.set_xscale('log')
        ax.set_xlabel('Parameters (M)')
        ax.set_ylabel('KL Divergence')
        ax.set_title(f'{arch}: KL Divergence', fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.15)

        # Panel C: Confidence drop
        ax = axes[row_idx, 2]
        for pert in perts:
            d = df_arch[df_arch['perturbation'] == pert].sort_values('size_M')
            c = pcols.get(pert, '#6B7280')
            ax.plot(d['size_M'], d['conf_drop'], 'o-', color=c,
                    linewidth=2, markersize=8, label=_pert_label(pert))
        ax.set_xscale('log')
        ax.set_xlabel('Parameters (M)')
        ax.set_ylabel('Confidence Drop')
        ax.set_title(f'{arch}: Confidence Drop', fontweight='bold')
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.15)

    fig.suptitle('Frozen Head Test Across Architectures',
                 fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/frozen_head_analysis.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()
    print(f'Saved {RESULTS_DIR}/frozen_head_analysis.png')
else:
    print('No frozen head results to plot.')

In [ ]:
# Combined Figure (Multi-Architecture)
#
# Procrustes ratio + Frozen Head results across all architectures.
# If external scaling CSVs exist (from individual experiment notebooks),
# we overlay the original scaling curves too.

from matplotlib.patches import Patch

archs_in_proc = sorted(proc_agg['arch'].unique())

fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))

# ---- Panel A: Procrustes Ratio (scaling curves, one line per arch) ----
ax = axes[0]
for arch in archs_in_proc:
    sub = proc_agg[proc_agg['arch'] == arch].sort_values('size_M')
    c = ARCH_COLORS.get(arch, '#6B7280')
    m = ARCH_MARKERS.get(arch, 'o')
    ax.plot(sub['size_M'], sub['ratio_mean'], marker=m, linestyle='-',
            color=c, linewidth=2.5, markersize=9, label=arch)
    ax.fill_between(sub['size_M'],
                    sub['ratio_mean'] - sub['ratio_std'],
                    sub['ratio_mean'] + sub['ratio_std'],
                    color=c, alpha=0.12)
ax.set_xscale('log')
ax.set_xlabel('Parameters (M)', fontsize=11)
ax.set_ylabel('Procrustes Ratio (aligned / raw)', fontsize=11)
ax.set_title('A. Procrustes Ratio vs. Scale', fontweight='bold', fontsize=12)
ax.axhline(y=1.0, color='#94A3B8', linestyle=':', linewidth=1, alpha=0.5)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.15)
ax.set_ylim(0, 1.15)

# ---- Panel B: Procrustes Ratio bars (all models grouped by arch) ----
ax = axes[1]
x_pos = 0
tick_positions, tick_labels_b = [], []
for arch in archs_in_proc:
    sub = proc_agg[proc_agg['arch'] == arch].sort_values('size_M')
    c = ARCH_COLORS.get(arch, '#6B7280')
    for _, row in sub.iterrows():
        ax.bar(x_pos, row['ratio_mean'], yerr=row['ratio_std'],
               color=c, edgecolor='white', linewidth=1.2, capsize=3, width=0.7, zorder=3)
        ax.text(x_pos, row['ratio_mean'] + row['ratio_std'] + 0.03,
                f'{row["ratio_mean"]:.2f}', ha='center', fontsize=7, fontweight='bold')
        tick_positions.append(x_pos)
        tick_labels_b.append(row['model'])
        x_pos += 1
    x_pos += 0.5
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels_b, fontsize=7, rotation=50, ha='right')
ax.set_ylabel('Procrustes Ratio', fontsize=11)
ax.set_title('B. All Models', fontweight='bold', fontsize=12)
ax.set_ylim(0, 1.15)
ax.axhline(y=1.0, color='#94A3B8', linestyle=':', linewidth=1, alpha=0.4)
ax.grid(True, alpha=0.15, axis='y')
ax.legend(handles=[Patch(facecolor=ARCH_COLORS.get(a, '#6B7280'), label=a)
                   for a in archs_in_proc], fontsize=9)

# ---- Panel C: Frozen Head KL (if available) ----
ax = axes[2]
if len(df_frozen) > 0:
    frozen_agg = df_frozen.groupby(['arch', 'model', 'size_M']).agg(
        agreement_mean=('agreement', 'mean'),
        kl_mean=('kl_divergence', 'mean'),
    ).reset_index().sort_values(['arch', 'size_M'])

    x_pos = 0
    tick_positions_c, tick_labels_c = [], []
    for arch in sorted(frozen_agg['arch'].unique()):
        sub = frozen_agg[frozen_agg['arch'] == arch]
        c = ARCH_COLORS.get(arch, '#6B7280')
        for _, row in sub.iterrows():
            ax.bar(x_pos, row['kl_mean'], color=c,
                   edgecolor='white', linewidth=1.2, width=0.7)
            tick_positions_c.append(x_pos)
            tick_labels_c.append(row['model'])
            x_pos += 1
        x_pos += 0.5
    ax.set_xticks(tick_positions_c)
    ax.set_xticklabels(tick_labels_c, fontsize=7, rotation=50, ha='right')
    ax.set_ylabel('KL Divergence (mean)', fontsize=11)
    ax.set_title('C. Frozen Head Test\n(higher = predictions diverge)', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.15, axis='y')
    ax.legend(handles=[Patch(facecolor=ARCH_COLORS.get(a, '#6B7280'), label=a)
                       for a in sorted(frozen_agg['arch'].unique())], fontsize=9)
else:
    ax.text(0.5, 0.5, 'Frozen Head test\nnot yet run', transform=ax.transAxes,
            ha='center', va='center', fontsize=14, color='#94A3B8')
    ax.set_title('C. Frozen Head Test', fontweight='bold', fontsize=12)

fig.suptitle('Phase Transition Proof: ESM-2, NT, Caduceus',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/phase_transition_proof.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved {RESULTS_DIR}/phase_transition_proof.png')